In [0]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","29","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","28","Male","60000","2013-04-01"],
    ["006","103","Mimi Wong","27","Female","52000","2018-07-01"],
    ["007","104","James Johnson","42","Male","70000","2012-03-15"],
    ["008","104","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","105","Tom Tan","31","Male","58000","2016-06-01"],
    ["010","105","Lisa Lee","27","Female","50000","2018-08-01"],
    ["011","106","David Park","34","Male","62000","2015-11-01"],
    ["012","106","Susan Chen","33","Female","54000","2017-02-15"],
    ["013","107","Brian Kim","36","Male","65000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","108","Michael Lee","40","Male","68000","2009-09-30"],
    ["016","108","Kelly Zhang","32","Female","53000","2018-04-01"],
    ["017","109","George Wang","29","Male","59000","2016-03-15"],
    ["018","109","Nancy Liu","28","Female","50000","2017-06-01"],
    ["019","110","Steven Chen","30","Male","60000","2015-08-01"],
    ["020","110","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"


df = spark.createDataFrame(data = emp_data, schema = emp_schema)

In [0]:
display(df)

In [0]:
#select max(salary) over(partiotion by emp_id order by salary desc) as new_col from table

from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy(df.employee_id).orderBy(df.salary.desc())
max_func = max(df.salary).over(window_spec)

df2 = df.withColumn("new_col",max_func)
display(df2)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy(col("department_id")).orderBy(col("salary").desc())
rn_func = row_number().over(window_spec)

df_rn = df.withColumn("rn",rn_func)

display(df_rn.where("rn = 2"))

In [0]:
display(df)

In [0]:
#display(df.orderBy(df.salary.desc()))

from pyspark.sql.functions import * 

df1 = df.groupBy("department_id").agg(count("employee_id").alias("count_employee_id"))
display(df1)

In [0]:
df3= df.groupBy("department_id").agg(sum("salary").alias("sumSal"))
display(df3)

In [0]:
df1 = df.where(df.salary > 50000).orderBy(df.employee_id)
display(df1)


In [0]:
df1.write.csv("/Volumes/workspace/databricks_prac/prac_1/emp_dup.csv", header = True)

In [0]:
df1.schema
#df1.printSchema()

In [0]:
from pyspark.sql.functions import *

df2 = df.select(df.employee_id,col("name"),expr("age"),df.salary)

In [0]:
display(df2)

new_df2 = df2.selectExpr ("employee_id as emp_id", "name", "cast(age as int) as age", "salary")

display(new_df2.printSchema())

In [0]:

from pyspark.sql.functions import *

df3 = df.select(expr("employee_id as emp_id"),df.name,expr("cast(age as int) as age"),df.salary)

display(df3.printSchema())



In [0]:
display(df3.where(df.age > 30))

In [0]:
display(df)

In [0]:
from pyspark.sql.functions import *
new_df1 = df.selectExpr("employee_id","name","age","cast(salary as double) as salary")
display(new_df1.printSchema())

In [0]:
df1_1 = new_df1.withColumn("tax",new_df1.salary*0.2)
display(df1_1)

In [0]:
from pyspark.sql.functions import *
df1_2 = df1_1.withColumn("column1",lit("1")).withColumn("column2",lit(2))
display(df1_2)

In [0]:
df1_3= df1_2.drop("column2","column1","tax")
display(df1_3.limit(5))



In [0]:
from pyspark.sql.functions import *
columns = {
"tax" : df1_3.salary*0.2,
"col1" : lit("1"),
"col2" : lit(2) 
}

df1_4 = df1_3.withColumns(columns)

In [0]:
display(df1_4.printSchema())

In [0]:
#display(df1)
df1_5 = df.withColumn("new_gender", when(df.gender == "Male","M").when(df.gender == "Female","F").otherwise(None))
display(df1_5)

In [0]:
from pyspark.sql.functions import *

df1_6 = df1_5.withColumn("new_gender2",expr("""case 
                                                when new_gender== 'M' then 'Male' 
                                                when new_gender == 'F' then 'Female' else "Null" end"""
                                            )
)

display(df1_6)

In [0]:
#display(df)

from pyspark.sql.functions import *

df_replace = df.withColumn("replace", regexp_replace(df.name," ","**")).withColumn("hire_date", to_date(df.hire_date)).withColumn("time",current_date()).withColumn("timestamp",current_timestamp())

display(df_replace)

In [0]:
df_cd = current_timestamp()
df_cd.show()